In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

file_path = "/content/Nassau Candy Distributor.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)
print("Column names:")
print(df.columns.tolist())


Dataset loaded successfully!
Rows and columns: (10194, 18)
Column names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost']


In [3]:

# Check missing values in each column
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())


Missing values in each column:
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
dtype: int64

Total missing values: 0


In [4]:
# Count completely duplicated rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

# Check whether Row ID values are unique
print("Unique Row IDs:", df["Row ID"].nunique())
print("Total rows:", len(df))


Number of duplicate rows: 0
Unique Row IDs: 10194
Total rows: 10194


In [5]:
# Convert dates into proper datetime format
df["Order Date"] = pd.to_datetime(
    df["Order Date"], dayfirst=True, errors="coerce"
)

df["Ship Date"] = pd.to_datetime(
    df["Ship Date"], dayfirst=True, errors="coerce"
)

# Check whether any dates failed to parse
print("Invalid Order Dates:", df["Order Date"].isna().sum())
print("Invalid Ship Dates:", df["Ship Date"].isna().sum())

# Check whether shipping happens before ordering
invalid_sequence = df["Ship Date"] < df["Order Date"]

print("Ship dates before order dates:", invalid_sequence.sum())

# Display the date ranges
print("\nOrder date range:")
print(df["Order Date"].min(), "to", df["Order Date"].max())

print("\nShip date range:")
print(df["Ship Date"].min(), "to", df["Ship Date"].max())


Invalid Order Dates: 0
Invalid Ship Dates: 0
Ship dates before order dates: 0

Order date range:
2024-01-02 00:00:00 to 2025-12-31 00:00:00

Ship date range:
2026-06-30 00:00:00 to 2030-06-28 00:00:00


In [6]:

# Calculate shipping lead time in days
df["Lead Time (Days)"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

print("Lead-time statistics:")
print(df["Lead Time (Days)"].describe())

print("\nShortest lead time:",
      df["Lead Time (Days)"].min(), "days")

print("Longest lead time:",
      df["Lead Time (Days)"].max(), "days")

print("Median lead time:",
      df["Lead Time (Days)"].median(), "days")


Lead-time statistics:
count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Lead Time (Days), dtype: float64

Shortest lead time: 904 days
Longest lead time: 1642 days
Median lead time: 1274.0 days


In [7]:

# Summarize lead time by shipping mode
lead_time_by_mode = (
    df.groupby("Ship Mode")["Lead Time (Days)"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(2)
)

print(lead_time_by_mode)


                count  min  median     mean   max
Ship Mode                                        
First Class      1548  905  1272.0  1338.28  1638
Same Day          547  904  1269.0  1333.44  1636
Second Class     1979  906  1273.0  1323.85  1640
Standard Class   6120  908  1274.0  1314.33  1642


In [8]:

# Check data types of important numeric columns
numeric_cols = ["Sales", "Units", "Gross Profit", "Cost"]

print("Data types:")
print(df[numeric_cols].dtypes)

print("\nSummary statistics:")
print(df[numeric_cols].describe())

print("\nZero or negative values:")
for col in numeric_cols:
    print(f"{col}: {(df[col] <= 0).sum()}")


Data types:
Sales           float64
Units             int64
Gross Profit    float64
Cost            float64
dtype: object

Summary statistics:
              Sales         Units  Gross Profit          Cost
count  10194.000000  10194.000000  10194.000000  10194.000000
mean      13.908537      3.791838      9.166451      4.742087
std       11.341020      2.228317      6.643740      5.061647
min        1.250000      1.000000      0.250000      0.600000
25%        7.200000      2.000000      4.900000      2.400000
50%       10.800000      3.000000      7.470000      3.600000
75%       18.000000      5.000000     12.250000      5.700000
max      260.000000     14.000000    130.000000    130.000000

Zero or negative values:
Sales: 0
Units: 0
Gross Profit: 0
Cost: 0


In [9]:

# Calculate the difference between Sales and Cost
df["Sales Minus Cost"] = df["Sales"] - df["Cost"]

# Compare with Gross Profit
df["Profit Difference"] = (
    df["Gross Profit"] - df["Sales Minus Cost"]
)

print("Rows where Gross Profit differs from Sales - Cost:",
      (df["Profit Difference"].abs() > 0.02).sum())

print("\nExample rows:")
print(
    df[["Sales", "Cost", "Gross Profit",
        "Sales Minus Cost", "Profit Difference"]].head(10)
)


Rows where Gross Profit differs from Sales - Cost: 0

Example rows:
   Sales  Cost  Gross Profit  Sales Minus Cost  Profit Difference
0   6.50  2.28          4.22              4.22      -8.881784e-16
1   7.50  2.60          4.90              4.90       0.000000e+00
2  10.47  3.00          7.47              7.47      -8.881784e-16
3  10.80  3.30          7.50              7.50      -8.881784e-16
4  11.25  3.90          7.35              7.35       0.000000e+00
5  32.40  9.90         22.50             22.50       0.000000e+00
6   7.50  2.60          4.90              4.90       0.000000e+00
7   9.75  3.42          6.33              6.33       0.000000e+00
8   6.98  2.00          4.98              4.98       0.000000e+00
9  13.00  4.56          8.44              8.44      -1.776357e-15


# Day 2 — Data Quality Validation and Cleaning

## 1. Dataset Overview
- Total records: 10,194
- Total columns: 18
- Missing values: 0
- Completely duplicated rows: 0
- Unique Row IDs: 10,194

## 2. Date Validation
- Invalid order dates: 0
- Invalid ship dates: 0
- Ship dates before order dates: 0
- Order date range: January 2024 – December 2025
- Ship date range: June 2026 – June 2030

## 3. Lead-Time Investigation
- Minimum lead time: 904 days
- Median lead time: 1,274 days
- Maximum lead time: 1,642 days
- Lead times are unusually long for the listed shipping modes.
- Median lead times are very similar across all four shipping modes.
- The cause of the date inconsistency has not been confirmed.

**Decision:** Do not use lead time for prediction until the date data is verified. Preserve the original dates.

## 4. Financial Data Validation
- Sales, Units, Gross Profit, and Cost have numeric data types.
- No zero or negative values were found in these columns.
- Gross Profit matches Sales minus Cost within a tolerance of 0.02 for all rows.

## 5. Day 2 Conclusion
The dataset passed basic checks for missing values, duplicate rows, date parsing, and financial consistency. However, the shipping dates raise a significant data-quality concern that requires further investigation.

No original rows were deleted, and no original date values were corrected.

